In [4]:
# Install required packages
!pip install -q torch transformers datasets accelerate peft trl bitsandbytes pandas

In [5]:
# Install Unsloth for efficient fine-tuning
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q xformers

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [6]:
import pandas as pd
import json
import random
from typing import List, Dict

import torch
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

MIN_TEST_N = 20

# ==========================================================================================================================
# CONFIGURATION
# ==========================================================================================================================

# Model settings
LOAD_IN_4BIT = False
MAX_SEQ_LENGTH = 2048

if LOAD_IN_4BIT:
    MODEL_NAME = "unsloth/llama-3-8b-bnb-4bit"
else:
    MODEL_NAME = "unsloth/llama-3-8b"

# Training hyperparameters
LEARNING_RATE = 2e-4
NUM_EPOCHS = 5
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4
WARMUP_STEPS = 5
SAVE_STEPS = 100
LOGGING_STEPS = 10

# LoRA configuration
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

OUTPUT_DIR = "./llama3-finetuned"


# ==========================================================================================================================
# DATA PREPARATION
# ==========================================================================================================================
def load_and_prepare_data(filename: str, test_ratio: float) -> tuple[List[Dict], List[Dict]]:
    """Load and split data into train/test sets with group-aware splitting."""
    df_train = pd.read_csv(filename)

    initial_count = df_train.shape[0]

    # Drop unused columns
    df_train = df_train.drop(columns=['judge1','judge2','judge3'])

    # Drop all rows for which yR is not null but the final decision is not yR
    df_train = df_train[(df_train['final_decision'] == 'yR') | (df_train['yR'].isnull())]
    print(f"Dropped {initial_count - df_train.shape[0]} rows")

    # Create formatted training data
    samples = []
    for idx, row in df_train.iterrows():
        samples.append({
            "id": idx,
            "prompt": row['prompt'],
            "yG": row['yG'],
            "c": row['c'] if not pd.isna(row['c']) else "",
            "yR": row['yR'] if not pd.isna(row['yR']) else ""
        })
    print(f"Created {len(samples)} formatted training examples")

    # Create test mask: each question is replicated 4 times with slight variations, so that
    # idx, idx+1, idx+2, idx+3 are all variations of the same question. If we take one for testing,
    # then all the others must be taken as well, otherwise we risk information leakage from training.
    num_groups = len(df_train) // 4

    # Randomly select which groups go to test set
    num_test_groups = max(int(num_groups * test_ratio), MIN_TEST_N // 4)
    test_group_indices = random.sample(range(num_groups), num_test_groups)

    # Convert group indices to actual row indices (each group has 4 consecutive rows)
    test_indices = []
    for group_idx in test_group_indices:
        base_idx = group_idx * 4
        test_indices.extend([base_idx, base_idx + 1, base_idx + 2, base_idx + 3])

    # Split the data
    test_samples = [samples[i] for i in test_indices]
    train_samples = [samples[i] for i in range(len(samples)) if i not in test_indices]

    print(f"Training set: {len(train_samples)} samples ({len(train_samples)//4} question groups)")
    print(f"Test set:     {len(test_samples)} samples ({len(test_samples)//4} question groups)")
    return train_samples, test_samples


# ==========================================================================================================================
# TRAINING FORMAT (Internal - teaches the refinement process)
# ==========================================================================================================================
def format_training_sample(sample):
    """
    Training format that teaches the model the self-correction process.

    Uses Llama 3's native chat format with an internal thought wrapper.
    The model learns to:
    1. Generate an initial response
    2. Critically evaluate it
    3. Produce a refined version

    This format is ONLY seen during training. The model internalizes the
    refinement process but won't output these markers at inference time
    because we'll use a different prompt structure then.
    """
    # Use the best available response for cases where yR might be empty
    best_response = sample['yR'] if sample['yR'] else sample['yG']
    critique = sample['c'] if sample['c'] else "This response is good as-is."

    text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a helpful assistant that generates responses, provides critical feedback, and revises responses to reduce sycophancy and improve accuracy.<|eot_id|><|start_header_id|>user<|end_header_id|>

{sample['prompt']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

<internal_thought>
Initial attempt: {sample['yG']}

Self-critique: {critique}

Refined response: {best_response}
</internal_thought>

{best_response}<|eot_id|>"""

    # Clean up formatting
    text = text.replace("\n\n\n", "\n\n").replace("  ", " ").replace("\t", " ")
    return {"text": text}


# ==========================================================================================================================
# INFERENCE FORMAT (User-facing - clean output only)
# ==========================================================================================================================
def create_inference_prompt(user_prompt: str) -> str:
    """
    Inference format that produces clean, user-facing output.

    The model has learned the refinement process internally during training,
    so when given a standard chat format at inference, it will:
    1. Mentally go through the refinement process (learned behavior)
    2. Output only the final polished response (what we want)

    No special tokens or internal markers appear in the output because
    they're not in this prompt structure.
    """
    return f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a helpful assistant that provides accurate, balanced responses without excessive agreement or flattery.<|eot_id|><|start_header_id|>user<|end_header_id|>

{user_prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""


# ==========================================================================================================================
# MODEL SETUP
# ==========================================================================================================================
def setup_model_and_tokenizer():
    """Load base model and add LoRA adapters."""
    print("Loading model and tokenizer...")
    print(f"Model: {MODEL_NAME}")
    print(f"4-bit quantization: {LOAD_IN_4BIT}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name      = MODEL_NAME,
        max_seq_length  = MAX_SEQ_LENGTH,
        dtype           = None,
        load_in_4bit    = LOAD_IN_4BIT,
        device_map      = "auto",
    )

    # Set proper padding token
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    print("Adding LoRA adapters...")
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )
    return model, tokenizer


# ==========================================================================================================================
# TRAINING
# ==========================================================================================================================
def prepare_datasets(train_samples, test_samples):
    """Convert samples to HuggingFace Dataset format with training template."""
    print(f"Preparing datasets...")
    print(f"Training samples: {len(train_samples)}")
    print(f"Test samples: {len(test_samples)}")

    train_dataset = Dataset.from_list(train_samples)
    test_dataset  = Dataset.from_list(test_samples)

    # Apply the training format (with internal thought process)
    train_dataset = train_dataset.map(format_training_sample, remove_columns=train_dataset.column_names)
    test_dataset  = test_dataset.map(format_training_sample, remove_columns=test_dataset.column_names)
    return train_dataset, test_dataset


def train_model(model, tokenizer, train_dataset, test_dataset):
    """Execute the training loop."""
    print("\nStarting training...")
    print(f"Epochs: {NUM_EPOCHS}")
    print(f"Batch size: {BATCH_SIZE}")
    print(f"Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")
    print(f"Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")

    trainer = SFTTrainer(
        model              = model,
        tokenizer          = tokenizer,
        train_dataset      = train_dataset,
        eval_dataset       = test_dataset,
        dataset_text_field = "text",
        max_seq_length     = MAX_SEQ_LENGTH,
        dataset_num_proc   = 2,
        packing            = False,
        args=TrainingArguments(
            per_device_train_batch_size = BATCH_SIZE,
            per_device_eval_batch_size  = BATCH_SIZE,
            gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS,
            warmup_steps                = WARMUP_STEPS,
            num_train_epochs            = NUM_EPOCHS,
            learning_rate               = LEARNING_RATE,
            fp16                        = not torch.cuda.is_bf16_supported(),
            bf16                        = torch.cuda.is_bf16_supported(),
            logging_steps               = LOGGING_STEPS,
            optim                       = "adamw_8bit",
            weight_decay                = 0.01,
            lr_scheduler_type           = "linear",
            seed                        = 3407,
            output_dir                  = OUTPUT_DIR,
            save_steps                  = SAVE_STEPS,
            eval_strategy               = "steps",
            eval_steps                  = SAVE_STEPS,
            save_total_limit            = 3,
            load_best_model_at_end      = True,
        ),
    )

    # Show training info
    gpu_stats = torch.cuda.get_device_properties(0)
    print(f"\nGPU: {gpu_stats.name}")
    print(f"GPU Memory: {round(gpu_stats.total_memory / 1024**3, 1)} GB")
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")

    # Train the model
    trainer_stats = trainer.train()
    print("\n" + "="*60)
    print("Training completed!")
    print("="*60)
    return trainer, trainer_stats


# ==========================================================================================================================
# SAVING
# ==========================================================================================================================
def save_model(model, tokenizer, output_dir=OUTPUT_DIR):
    """Save the fine-tuned model - LoRA adapters only for reliability."""
    import os

    print(f"\nSaving model to {output_dir}...")

    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Save LoRA adapters (most reliable method)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"✓ LoRA adapters saved to: {output_dir}")

    print("\nModel saved successfully!")
    print(f"To load later, use: FastLanguageModel.from_pretrained('{output_dir}')")


# ==========================================================================================================================
# MAIN EXECUTION
# ==========================================================================================================================
if __name__ == "__main__":
    import os
    print("="*60)
    print("Llama 3 8B Self-Correction Fine-tuning")
    print("="*60)

    # Load data
    train_samples, test_samples = load_and_prepare_data("Llama3_answers.csv", 0.1)

    # Setup
    model, tokenizer = setup_model_and_tokenizer()
    train_dataset, test_dataset = prepare_datasets(train_samples, test_samples)

    # Train
    trainer, trainer_stats = train_model(model, tokenizer, train_dataset, test_dataset)

    # Save the model
    save_model(model, tokenizer)

    print("\n" + "="*60)
    print("Training pipeline complete!")
    print("="*60)
    print("\nNext steps:")

Llama 3 8B Self-Correction Fine-tuning
Dropped 42 rows
Created 958 formatted training examples
Training set: 866 samples (216 question groups)
Test set:     92 samples (23 question groups)
Loading model and tokenizer...
Model: unsloth/llama-3-8b
4-bit quantization: False
==((====))==  Unsloth 2025.12.1: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
100,0.453800,0.625791


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


eval/loss,▁
eval/runtime,▁
eval/samples_per_second,▁
eval/steps_per_second,▁
train/epoch,▁▂▂▃▃▄▄▅▅▆▆▆▇▇██
train/global_step,▁▂▂▃▃▄▄▅▅▆▆▆▇▇██
train/grad_norm,▆▃▂▁▁▁▁▁▁▁▂▂▂█
train/learning_rate,█▇▇▆▆▅▅▄▄▃▃▂▂▁
train/loss,█▃▂▂▂▂▂▂▁▁▁▁▁▁
eval/loss,0.62579
eval/runtime,6.5716



Training completed!

Saving model to ./llama3-finetuned...
✓ LoRA adapters saved to: ./llama3-finetuned

Model saved successfully!
To load later, use: FastLanguageModel.from_pretrained('./llama3-finetuned')

Training pipeline complete!

Next steps:


In [1]:
# ==========================================================================================================================
# INFERENCE TESTING
# ==========================================================================================================================
import csv
import os
import json
import pandas as pd
from tqdm import tqdm
import torch
from unsloth import FastLanguageModel

# Configuration from training script
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = False
BASE_MODEL_NAME = "unsloth/llama-3-8b"
FINETUNED_MODEL_PATH = "./llama3-finetuned"


def create_inference_prompt(user_prompt: str) -> str:
    """
    Inference format that produces clean, user-facing output.
    (Copied from training script to ensure consistency)
    """
    return f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a helpful assistant that provides accurate, balanced responses without excessive agreement or flattery.<|eot_id|><|start_header_id|>user<|end_header_id|>

{user_prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""


def test_inference(model, tokenizer, test_prompt: str, model_name: str, metadata: dict = None):
    """
    Test the model with clean, user-facing prompt format.
    """
    # Ensure model is in inference mode
    model.eval()

    # Use the inference format (clean, no internal markers)
    prompt = create_inference_prompt(test_prompt)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # Generate the response
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode the response
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract just the assistant's response (everything after the prompt)
    assistant_response = response.split("<|start_header_id|>assistant<|end_header_id|>")[-1].strip()
    if "<|eot_id|>" in assistant_response:
        assistant_response = assistant_response.split("<|eot_id|>")[0].strip()
    return assistant_response


def run_inference_tests(test_file, finetuned_model=None, finetuned_tokenizer=None):
    """
    Main function to run inference tests.
    Can accept pre-loaded models or load them fresh.
    """
    print("\n" + "="*60)
    print(f"Running Inference Tests on: {test_file}")
    print("="*60)

    # Read the JSONL file (each line is a separate JSON object)
    questions_data = []
    with open(test_file, "r") as f:
        for line in f:
            data = json.loads(line.strip())
            # Extract the question from the prompt
            question_text = data["base"]["question"]
            questions_data.append({
                "prompt": question_text,
                "correct_answer": data["base"]["correct_answer"],
            })

    print(f"Loaded {len(questions_data)} questions from file")

    # Create result csv file with header
    output_file = f"inference_results_{test_file.replace('.jsonl', '.csv')}"
    with open(output_file, "w", newline='', encoding='utf-8') as f:
        writer = csv.writer(f, quoting=csv.QUOTE_ALL)
        writer.writerow(["prompt", "correct_answer", "base_response", "finetuned_response"])

    # Load the original base model
    print("\nLoading original base model...")
    base_model, base_tokenizer = FastLanguageModel.from_pretrained(
        model_name     = BASE_MODEL_NAME,
        max_seq_length = MAX_SEQ_LENGTH,
        dtype          = None,
        load_in_4bit   = LOAD_IN_4BIT,
        device_map     = "auto",
    )

    # Set proper padding token
    base_tokenizer.pad_token = base_tokenizer.eos_token
    base_tokenizer.padding_side = "right"

    # Set the model in inference mode
    FastLanguageModel.for_inference(base_model)
    print("✓ Base model loaded successfully")

    # Test base model
    print("\nTesting Base Model...")
    base_results = []
    for question in tqdm(questions_data, desc="Base Model Inference"):
        base_results.append(test_inference(
            base_model,
            base_tokenizer,
            question['prompt'],
            "base_model",
            metadata=question
        ))

    # Clean up base model from memory
    del base_model
    del base_tokenizer
    torch.cuda.empty_cache()
    print("✓ Base model cleared from memory")

    # Load or use the fine-tuned model
    print("\n" + "="*60)

    # Check if models were passed in (from training script)
    if finetuned_model is not None and finetuned_tokenizer is not None:
        print("Using fine-tuned model from memory (passed from training)")
        ft_model = finetuned_model
        ft_tokenizer = finetuned_tokenizer
    else:
        # Load from disk
        print("Loading fine-tuned model from disk...")
        try:
            # First, load the base model
            ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
                model_name     = BASE_MODEL_NAME,
                max_seq_length = MAX_SEQ_LENGTH,
                dtype          = None,
                load_in_4bit   = LOAD_IN_4BIT,
                device_map     = "auto",
            )

            # Then load the LoRA adapters
            from peft import PeftModel
            ft_model = PeftModel.from_pretrained(ft_model, FINETUNED_MODEL_PATH)

            print("✓ Fine-tuned model loaded from disk successfully")
        except Exception as e:
            print(f"❌ Error loading fine-tuned model: {e}")
            print("Please make sure the model was saved correctly.")
            return

    # Set proper padding token
    ft_tokenizer.pad_token = ft_tokenizer.eos_token
    ft_tokenizer.padding_side = "right"

    FastLanguageModel.for_inference(ft_model)
    print("✓ Fine-tuned model ready for inference")

    # Test fine-tuned model
    print("\nTesting Fine-tuned Model...")
    finetuned_results = []
    for question in tqdm(questions_data, desc="Fine-tuned Model Inference"):
        finetuned_results.append(test_inference(
            ft_model,
            ft_tokenizer,
            question['prompt'],
            "finetuned_model",
            metadata=question
        ))

    print("\n✓ Inference testing complete!")

    # Store results in the csv file
    print(f"\nWriting results to {output_file}...")
    with open(output_file, "a", newline='', encoding='utf-8') as f:
        writer = csv.writer(f, quoting=csv.QUOTE_ALL)
        for i in range(len(questions_data)):
            writer.writerow([
                questions_data[i]['prompt'],
                questions_data[i]['correct_answer'],
                base_results[i],
                finetuned_results[i]
            ])

    print(f"✓ Results saved to: {output_file}")
    print("="*60)


if __name__ == "__main__":
    # Try to use models from the global scope (if run after training in same session)
    try:
        test_files = ["questions_syc.jsonl"]
        for test_file in test_files:
            # These variables should exist if training script was run in the same session
            run_inference_tests(test_file, model, tokenizer)
    except NameError:
        # If not found, load from disk
        print("Note: Models not found in memory. Will load from disk.")
        test_files = ["questions_syc.jsonl"]
        for test_file in test_files:
            run_inference_tests(test_file)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Note: Models not found in memory. Will load from disk.

Running Inference Tests on: questions_syc.jsonl
Loaded 21 questions from file

Loading original base model...
==((====))==  Unsloth 2025.12.1: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✓ Base model loaded successfully

Testing Base Model...


Base Model Inference: 100%|██████████| 21/21 [04:48<00:00, 13.72s/it]


✓ Base model cleared from memory

Loading fine-tuned model from disk...
==((====))==  Unsloth 2025.12.1: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✓ Fine-tuned model loaded from disk successfully
✓ Fine-tuned model ready for inference

Testing Fine-tuned Model...


Fine-tuned Model Inference: 100%|██████████| 21/21 [09:08<00:00, 26.10s/it]


✓ Inference testing complete!

Writing results to inference_results_questions_syc.csv...
✓ Results saved to: inference_results_questions_syc.csv
